# Full Document Transduction: GLM-OCR (Ollama) vs. Docling
## End-to-End Ingestion of `jpmc_annualreport-2025.pdf` (364 Pages)

In this notebook, we run and evaluate full-document visual transduction using **`glm-ocr:latest`** running locally on our **RTX 4070 Ti SUPER (16 GB VRAM)** via **Ollama**.

### Key Objectives:
1. **Full-Document Pipeline Monitoring:** Track live progress, per-page latency, and checkpoint statistics across all 364 pages.
2. **Head-to-Head Comparison with Docling:** Directly compare extracted Markdown on the hardest benchmark pages:
   - **Page 2:** Borderless Financial Highlights Table
   - **Page 8:** Pure Vector Chart (20-year net income progression)
   - **Page 16:** 2-Column Shareholder Letter
   - **Page 76:** Three-Year Summary of Consolidated Financial Highlights
   - **Page 85:** Provision for Credit Losses Table
3. **Consolidated Markdown Assembly:** Inspect the complete structured document ready for multimodal vector indexing.


In [ ]:
import sys
import json
import time
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path("..").resolve()
OUTPUT_DIR = PROJECT_ROOT / "parsing" / "output" / "glm_full"
CHECKPOINT_PATH = OUTPUT_DIR / "checkpoint.json"
PAGES_DIR = OUTPUT_DIR / "pages"
DOCLING_JSON = PROJECT_ROOT / "parsing" / "output" / "full_docling_parsed.json"

print(f"GLM Output directory: {OUTPUT_DIR}")
print(f"Checkpoint exists: {CHECKPOINT_PATH.exists()}")
print(f"Docling parsed JSON exists: {DOCLING_JSON.exists()}")


### 1. Ingestion Progress & Latency Telemetry
Inspect the real-time status of the batch ingestion job.


In [ ]:
if CHECKPOINT_PATH.exists():
    with open(CHECKPOINT_PATH, "r", encoding="utf-8") as f:
        ckpt = json.load(f)
        
    records = []
    for p_str, data in ckpt.items():
        records.append({
            "page": int(p_str),
            "status": data.get("status"),
            "chars": data.get("chars", 0),
            "elapsed_s": data.get("elapsed_s", 0)
        })
    df_prog = pd.DataFrame(records).sort_values("page")
    
    total_completed = len(df_prog[df_prog.status == "done"])
    total_chars = df_prog["chars"].sum()
    avg_latency = df_prog["elapsed_s"].mean()
    est_total_min = (364 * avg_latency) / 60
    
    print(f"--- Ingestion Status ---")
    print(f"Pages Completed: {total_completed} / 364 ({total_completed / 364 * 100:.1f}%)")
    print(f"Total Extracted Characters: {total_chars:,}")
    print(f"Average Latency per Page:   {avg_latency:.2f} seconds")
    print(f"Estimated Full Run Time:    {est_total_min:.1f} minutes")
    
    display(df_prog.tail(10))
else:
    print("No checkpoint found yet. Start the runner via: python parsing/scripts/run_full_glm_ocr.py")


### 2. Head-to-Head Comparison: Page 2 (Borderless Financial Highlights)
Compare the extracted table representation between **Docling** and **GLM-OCR**.


In [ ]:
p2_file = PAGES_DIR / "page_002.md"
if p2_file.exists():
    glm_p2_text = p2_file.read_text(encoding="utf-8")
    print("=== GLM-OCR Extracted Markdown (Page 2) ===")
    print(glm_p2_text[:800])
else:
    print(f"Page 2 not yet generated at {p2_file}")


### 3. Head-to-Head Comparison: Page 8 (Pure Vector Financial Chart)
Compare how **Docling** (bounding box detection) and **GLM-OCR** (semantic transcription) handle visual charts.


In [ ]:
p8_file = PAGES_DIR / "page_008.md"
if p8_file.exists():
    glm_p8_text = p8_file.read_text(encoding="utf-8")
    print("=== GLM-OCR Extracted Markdown (Page 8 Chart) ===")
    print(glm_p8_text[:1000])
else:
    print(f"Page 8 not yet generated at {p8_file}")


### 4. Consolidated Markdown Document Inspection
Check the final assembled markdown file ready for vector store chunking.


In [ ]:
full_md = OUTPUT_DIR / "full_jpmc_glm_ocr.md"
if full_md.exists():
    size_mb = full_md.stat().st_size / (1024 * 1024)
    print(f"Full consolidated markdown exists! Size: {size_mb:.2f} MB")
    content = full_md.read_text(encoding="utf-8")
    print(f"Total lines: {len(content.splitlines()):,}")
else:
    print(f"Full markdown not yet assembled at {full_md}")
